# 2D PINN: Glomerular Filtration Simulation
This notebook acts as an interactive walkthrough of the 2D Physics-Informed Neural Network (PINN) training and visualization. The core physics engine and neural network architecture are cleanly separated into the `src/` directory to maintain software engineering standards.

## 0. Environment Setup
Run this cell to ensure all required libraries are installed directly into the notebook's active kernel.

In [ ]:
%pip install torch numpy matplotlib scipy

In [8]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.train import train_pinn

# Ensure plots display inline
%matplotlib inline


## 1. Normotensive State (Healthy Simulation)
We train the model with normal hydrostatic pressure ($u_{max}=1.0$) and a healthy filtration coefficient ($k=1.5$).

In [ ]:
print("[+] Training Normotensive Model...")
model_healthy = train_pinn(epochs=6000, lr=1e-3, u_max=1.0, D=0.01, k=1.5)

[+] Training Normotensive Model...
Training 2D PINN on device: cpu
Epoch 00000 | Total: 40.8803 | Data: 0.2043 | Physics: 0.0231
Epoch 00500 | Total: 4.0091 | Data: 0.0176 | Physics: 0.4862
Epoch 01000 | Total: 2.0036 | Data: 0.0081 | Physics: 0.3787
Epoch 01500 | Total: 1.4183 | Data: 0.0062 | Physics: 0.1803
Epoch 02000 | Total: 1.0255 | Data: 0.0045 | Physics: 0.1200
Epoch 02500 | Total: 0.8373 | Data: 0.0037 | Physics: 0.1040
Epoch 03000 | Total: 0.7042 | Data: 0.0031 | Physics: 0.0825
Epoch 03500 | Total: 0.5909 | Data: 0.0026 | Physics: 0.0658
Epoch 04000 | Total: 0.5086 | Data: 0.0023 | Physics: 0.0527
Epoch 04500 | Total: 0.4210 | Data: 0.0019 | Physics: 0.0371


### Visualize Healthy Solute Clearance

In [ ]:
def plot_2d_snapshot(model, title):
    model.eval()
    x = np.linspace(0, 1, 100)
    y = np.linspace(0, 1, 100)
    X, Y = np.meshgrid(x, y)
    x_flat = torch.tensor(X.flatten()[:, None], dtype=torch.float32)
    y_flat = torch.tensor(Y.flatten()[:, None], dtype=torch.float32)
    t_flat = torch.ones_like(x_flat) # Snapshot at t=1.0
    
    with torch.no_grad():
        C_pred = model(x_flat, y_flat, t_flat).numpy()
        
    C_grid = C_pred.reshape(X.shape)
    
    plt.figure(figsize=(10, 4))
    contour = plt.contourf(X, Y, C_grid, 60, cmap='turbo')
    plt.colorbar(contour, label='Concentration C(x,y)')
    plt.xlabel('Capillary Length (x)')
    plt.ylabel('Capillary Width (y)')
    plt.title(title)
    plt.show()

plot_2d_snapshot(model_healthy, "Normotensive State (Effective Solute Clearance)")

## 2. Hypertensive State (Pathological Simulation)
We train the model with high hydrostatic pressure ($u_{max}=3.5$) and impaired filtration ($k=0.1$).

In [ ]:
print("[+] Training Hypertensive Model...")
model_diseased = train_pinn(epochs=6000, lr=1e-3, u_max=3.5, D=0.01, k=0.1)

plot_2d_snapshot(model_diseased, "Hypertensive State (Impaired Filtration)")